### Stream Processing

Processing  streams efficiently requires tools and frameworks designed to handle large volumes of data as it arrives. When it comes to processing Kafka streams, you have several options:

- **Native Kafka Consumers**: Writing your own consumer using the native Kafka library allows for custom processing logic.
- **Apache Kafka Streams**: Kafka Streams is a client library for building applications and microservices where the input and output data are stored in Kafka clusters. Unfortunately, it exists only in Java, which may not be ideal.
- **Apache Spark**: Utilizing Spark's streaming capabilities provides robust, distributed processing power.
- **Apache Flink**: Also a stream processing framework that additionally provides windowing and state management. Flink’s Python API, PyFlink, allows you to write your stream processing jobs in Python, but it's wrapper over the Flink Java APIs and rather difficult to set up.

However, since we primarily work with the Python ecosystem, Faust presents a modern and compelling option.

#### Faust

Faust is a kafka stream processing library, built on Python, designed to process large volumes of data in real time.
It was developed by Robinhood and is heavily inspired by Kafka Streams. It can be integrated with Kafka or used as a standalone framework.

Faust provides a Python-centric approach to stream processing. It also allows for typed processing of streams.
e.g., Taking into account "our music event stream", we can build a class in faust, that illustrates each message and allows for analyzing it:


##### Records in Faust

The Record class in Faust is crucial for defining the schema of the data that flows through your stream processing application. It models the data as Python classes, which makes the data easier to work with. By defining each field, you ensure that each instance of the record can be automatically serialized and deserialized from and to JSON (or other formats), which is essential for message passing in Kafka.

**Type Safety**: Faust’s use of the Record class helps introduce type safety to stream processing. Type annotations ensure that your application behaves as expected, reducing runtime errors and improving reliability. When you define a Record, you specify types for each field, which Faust enforces.

In [ ]:
%%file faust_music_events.py

import faust

class MusicEvent(faust.Record):
    ts: int
    auth: str
    page: str
    song: str
    level: str
    artist: str
    gender: str
    method: str
    status: int
    userId: str
    lastName: str
    location: str
    track_id: int
    firstName: str
    sessionId: int
    userAgent: str
    registration: int
    itemInSession: int

### State Management in Stream Processing

State management is a critical feature in stream processing systems, allowing for the maintenance of state across events for complex event processing, aggregations, and other operations that require historical context.

#### State Store in Faust

Faust provides a built-in mechanism for state management called the *state store*. This state store can be used to persist the state of stream processing tasks, which is essential for ensuring fault tolerance and enabling stateful computations.

**RocksDB Integration**:
- **What is RocksDB?**: RocksDB is an embeddable persistent key-value store for fast storage, developed by Facebook based on LevelDB. It's optimized for fast, low-latency storage of data.
- **Integration in Faust**: Faust integrates seamlessly with RocksDB to provide a local disk-based state store. This integration enables Faust applications to handle large states efficiently, with quick access times that are crucial for real-time processing needs.
- **Benefits**: Using RocksDB with Faust allows applications to recover quickly from failures by reloading the state from disk. It also supports state larger than what can be held in memory, which is beneficial for processing large datasets or long window operations.

**Redis Integration**:
- **What is Redis?**: Redis is an in-memory data structure store, used as a database, cache, and message broker. It supports various data structures such as strings, hashes, lists, sets, and sorted sets.
- **Integration in Faust**: Faust can also integrate with Redis to manage state. Redis is beneficial for scenarios where quick access to state is necessary and persistence can be handled through Redis' snapshotting and replication features.
- **Benefits**: Using Redis, Faust applications can achieve fast read and write operations with the ability to scale horizontally by distributing the state across multiple Redis instances.

**Alternative State Stores**:
- **Memory**: For less complex or smaller-scale applications where the state does not need to survive restarts, Faust can manage state entirely in memory. This method provides faster access but lacks persistence.
- **Custom Stores**: Developers can integrate other databases or key-value stores as state backends if specific requirements, such as distributed state management or multi-node synchronization, are necessary

#### Initialize the Faust App

The Faust application initialization involves setting up a standalone application that can process streaming data. This is critical as it establishes the operational context including connection to Kafka, serialization, and state management.


```python
app = faust.App('music_stream_processor',
                topic_replication_factor=3,
                topic_partitions=1,
                broker=f"kafka://{kafka_app_config['bootstrap.servers']}",
                value_serializer='json',
                store='rocksdb://',
                broker_credentials=creds)

````

#### Configure a Kafka Topic and the Record

Specify the name and type of data the topic will handle. 
Here, MusicEvent is a Faust record, which means every message in this topic should conform to the structure defined by the MusicEvent class.
This ensures that all messages are serialized and deserialized properly

```python
topic = app.topic('music_streams', value_type=MusicEvent)
```

## Managing State with Faust Table

Configure a table (state store). Tables in Faust act like in-memory key-value stores but are optimized for stream processing.
We'll add a `song_plays` table  designed to manage the state of song play counts per user:

```python
song_plays = app.Table('song_plays', default=int)
````

#### Define a stream processor

A stream processor is the heart of a Faust application. It defines how to process events from a Kafka topic. 
**app.agent** You can define a stream processor by using the `@app.agent` decorator. The `@app.agent` decorator takes a Kafka topic as an argument. The stream processor will process events from this topic.

The stream processor is an asynchronous function that takes a stream of events as an argument.
It can process events from the stream using a `for` loop and also updates our state store. As you can see here in this example below, 
we are updating the `song_plays` table with the number of songs listened to by each user.
We are also printing the number of songs listened to by each user to the console.

```python
@app.agent(topic)
async def process(stream):
    async for event in stream:
        song_plays[event.userId] += 1
        print(f'User {event.userId} has listened to {song_plays[event.userId]} songs.')
```

Put it all together, create your faus_app.py with the following content:

In [ ]:
%%file faust_app.py

import faust
from faust.types.auth import AuthProtocol
import ssl
from utils import ccloud_lib
from faust_music_events import MusicEvent

# Read the Kafka configuration
kafka_app_config = ccloud_lib.read_ccloud_config("kafka.config")

# Set up SASL credentials
creds = faust.SASLCredentials(
    username=kafka_app_config['sasl.username'],
    password=kafka_app_config['sasl.password'],
    mechanism='PLAIN',
    ssl_context=ssl.create_default_context()
)

# Initialize the Faust app
app = faust.App('music_stream_processor',
                topic_replication_factor=3,
                topic_partitions=1,
                broker=f"kafka://{kafka_app_config['bootstrap.servers']}",
                value_serializer='json',
                store='rocksdb://',
                broker_credentials=creds)

# Define a Kafka topic with MusicEvent as the value type
topic = app.topic('music_streams', value_type=MusicEvent)
song_plays = app.Table('song_plays', default=int)

# Define a stream processor
@app.agent(topic)
async def process(stream):
    async for event in stream:
        song_plays[event.userId] += 1
        print(f'User {event.userId} has listened to {song_plays[event.userId]} songs.')

### Run the Faust app

In order to run the faust app, you'll need to run the following command in the terminal:

```bash
faust -A faust_app worker -l info
```

This command will start the Faust app and begin processing the incoming music events. 

```sh
faust -A faust_app  worker -l info
```

### Processing of events in the console

You should now see that our faust agent is triggering our event_processor. For each incoming event, the songplays per user are being updated. You'll also see, that the state is persistent. You can now stop the app using Ctrl+C in the terminal.

#### Lets do something more interesting

In addition to printing the counts to the console, we can expose the song play counts through a REST API using Faust's web server capabilities. This allows us to query the counts for individual users or all users via HTTP endpoints.

```python
@app.page('/counts/{userId}/')
async def get_count(self, request, userId):
    # Retrieve the song play count for a specific user
    count = song_plays[userId]
    return app.web.json({'userId': userId, 'count': count})

@app.page('/counts/')
async def get_all_counts(self, request):
    # Create a dictionary to store all user counts
    all_counts = {user_id: count for user_id, count in song_plays.items()}
    return app.web.json(all_counts)

**The new app providing the REST API Endpoints would then look like this**

When running this script inside your terminal, use the app file name `faust_app_web.py`.

Wait until the logs show up and then visit the url 
*codespace-url/counts/* 

You should see all registered songplays and and counts per user - you can also visit *codespace-url*/counts/1370/ - to have events filtered by user

In [ ]:
%%file faust_app_web.py

import faust
from faust.types.auth import AuthProtocol
import ssl
from utils import ccloud_lib
from faust_music_events import MusicEvent

# Read the Kafka configuration
kafka_app_config = ccloud_lib.read_ccloud_config("kafka.config")

# Set up SASL credentials
creds = faust.SASLCredentials(
    username=kafka_app_config['sasl.username'],
    password=kafka_app_config['sasl.password'],
    mechanism='PLAIN',
    ssl_context=ssl.create_default_context()
)

# Initialize the Faust app
app = faust.App('music_stream_processor',
                topic_replication_factor=3,
                topic_partitions=1,
                broker=f"kafka://{kafka_app_config['bootstrap.servers']}",
                value_serializer='json',
                store='rocksdb://',
                broker_credentials=creds)

# Define a Kafka topic with MusicEvent as the value type
topic = app.topic('music_streams', value_type=MusicEvent)
song_plays = app.Table('song_plays', default=int)

# Define a stream processor
@app.agent(topic)
async def process(stream):
    async for event in stream:
        song_plays[event.userId] += 1
        print(f'User {event.userId} has listened to {song_plays[event.userId]} songs.')


@app.page('/counts/{userId}/')
async def get_count(self, request, userId):
    count = song_plays[userId]
    return app.web.json({'userId': userId, 'count': count})

@app.page('/counts/')
async def get_all_counts(self, request):
    # Create a dictionary to store all user counts
    all_counts = [{'userId': user_id, 'songplays': count} for user_id, count in song_plays.items()]
    return app.web.json(all_counts)


#### One more example with artist counts

In this last example, we'll also count per artist

In [ ]:
%%file faust_app_web_artist.py

import faust
import ssl
from utils import ccloud_lib
from faust_music_events import MusicEvent

# Read the Kafka configuration
kafka_app_config = ccloud_lib.read_ccloud_config("kafka.config")

# Set up SASL credentials
creds = faust.SASLCredentials(
    username=kafka_app_config['sasl.username'],
    password=kafka_app_config['sasl.password'],
    mechanism='PLAIN',
    ssl_context=ssl.create_default_context()
)

# Initialize the Faust app
app = faust.App('music_stream_processor',
                topic_replication_factor=3,
                topic_partitions=1,
                broker=f"kafka://{kafka_app_config['bootstrap.servers']}",
                value_serializer='json',
                store='rocksdb://',
                broker_credentials=creds)

# Define a Kafka topic with MusicEvent as the value type
topic = app.topic('music_streams', value_type=MusicEvent)
song_plays = app.Table('song_plays', default=int)
artist_plays = app.Table('artist_plays', default=int)

# Define a stream processor
@app.agent(topic)
async def process(stream):
    async for event in stream:
        song_plays[event.userId] += 1
        artist_plays[event.artist] += 1
        print(f'User {event.userId} has listened to {song_plays[event.userId]} songs.')
        print(f'Artist {event.artist} has been played {artist_plays[event.artist]} times.')

@app.page('/counts/{userId}/')
async def get_count(self, request, userId):
    count = song_plays[userId]
    return app.web.json({'userId': userId, 'count': count})

@app.page('/counts/')
async def get_all_counts(self, request):
    # Create a dictionary to store all user counts
    all_counts = [{'userId': user_id, 'songplays': count} for user_id, count in song_plays.items()]
    return app.web.json(all_counts)

@app.page('/artist_counts/')
async def get_artist_counts(self, request):
    # Create a list to store artist counts in the desired format
    artist_counts = [{'artist': artist, 'songplays': count} for artist, count in artist_plays.items()]
    return app.web.json(artist_counts)